<a href="https://colab.research.google.com/github/DanishShah619/git_agent/blob/main/git_man.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone --depth 1 https://github.com/git/git.git


fatal: destination path 'git' already exists and is not an empty directory.


In [2]:
import os
doc_files = [f for f in os.listdir('git/Documentation') if f.startswith('git-') and f.endswith('.adoc')]
print(len(doc_files), "command doc files found")
print(doc_files[:10])

169 command doc files found
['git-rm.adoc', 'git-rebase.adoc', 'git-request-pull.adoc', 'git-unpack-file.adoc', 'git-difftool.adoc', 'git-check-attr.adoc', 'git-prune-packed.adoc', 'git-verify-commit.adoc', 'git-show.adoc', 'git-branch.adoc']


In [3]:
with open('git/Documentation/git-rebase.adoc', 'r') as f:
    content = f.read()

print(content[:2000])

git-rebase(1)

NAME
----
git-rebase - Reapply commits on top of another base tip

SYNOPSIS
--------
[verse]
'git rebase' [-i | --interactive] [<options>] [--exec <cmd>]
	[--onto <newbase> | --keep-base] [<upstream> [<branch>]]
'git rebase' [-i | --interactive] [<options>] [--exec <cmd>] [--onto <newbase>]
	--root [<branch>]
'git rebase' (--continue|--skip|--abort|--quit|--edit-todo|--show-current-patch)

DESCRIPTION
-----------
Transplant a series of commits onto a different starting point.
You can also use `git rebase` to reorder or combine commits: see INTERACTIVE
MODE below for how to do that.

For example, imagine that you have been working on the `topic` branch in this
history, and you want to "catch up" to the work done on the `master` branch.

------------
          A---B---C topic
         /
    D---E---F---G master
------------

You want to transplant the commits you made on `topic` since it diverged from
`master` (i.e. A, B, and C), on top of the current `master`.  You can do

In [4]:
import re

def parse_adoc_file(filepath):
    with open(filepath, 'r', errors='ignore') as f:
        text = f.read()

    cmd_name = os.path.basename(filepath).replace('.adoc', '')

    # Match: an all-caps header line, followed by a line of dashes
    pattern = re.compile(r'^([A-Z][A-Z \-]{2,})\n-{3,}\n', re.MULTILINE)

    matches = list(pattern.finditer(text))
    chunks = []

    for i, m in enumerate(matches):
        header = m.group(1).strip()
        start = m.end()
        end = matches[i+1].start() if i + 1 < len(matches) else len(text)
        body = text[start:end].strip()
        if body:
            chunks.append({
                "text": f"{header}\n{body}",
                "metadata": {"command": cmd_name, "section": header}
            })

    return chunks

In [5]:
chunks = parse_adoc_file('git/Documentation/git-rebase.adoc')
print(len(chunks), "chunks found")
for c in chunks:
    print("---", c['metadata']['section'], "---")
    print(c['text'][:150])
    print()

15 chunks found
--- NAME ---
NAME
git-rebase - Reapply commits on top of another base tip

--- SYNOPSIS ---
SYNOPSIS
[verse]
'git rebase' [-i | --interactive] [<options>] [--exec <cmd>]
	[--onto <newbase> | --keep-base] [<upstream> [<branch>]]
'git rebase' [

--- DESCRIPTION ---
DESCRIPTION
Transplant a series of commits onto a different starting point.
You can also use `git rebase` to reorder or combine commits: see INTERACTI

--- TRANSPLANTING A TOPIC BRANCH WITH --ONTO ---
TRANSPLANTING A TOPIC BRANCH WITH --ONTO
Here is how you would transplant a topic branch based on one
branch to another, to pretend that you forked th

--- MODE OPTIONS ---
MODE OPTIONS
The options in this section cannot be used with any other option,
including not with each other:

--continue::
	Restart the rebasing proc

--- OPTIONS ---
OPTIONS
--onto <newbase>::
	Starting point at which to create the new commits. If the
	`--onto` option is not specified, the starting point is
	`<upst

--- INCOMPATIBLE OPTIONS -

In [6]:
SKIP_SECTIONS = {"GIT"}  # boilerplate footer, no useful content

def parse_adoc_file(filepath):
    with open(filepath, 'r', errors='ignore') as f:
        text = f.read()

    cmd_name = os.path.basename(filepath).replace('.adoc', '')

    # Match: an all-caps header line, followed by a line of dashes
    pattern = re.compile(r'^([A-Z][A-Z \-]{2,})\n-{3,}\n', re.MULTILINE)

    matches = list(pattern.finditer(text))
    chunks = []

    for i, m in enumerate(matches):
        header = m.group(1).strip()
        if header in SKIP_SECTIONS:
            continue
        start = m.end()
        end = matches[i+1].start() if i + 1 < len(matches) else len(text)
        body = text[start:end].strip()
        if body:
            chunks.append({
                "text": f"{header}\n{body}",
                "metadata": {"command": cmd_name, "section": header}
            })

    return chunks

In [7]:
all_chunks = []
for f in doc_files:
    all_chunks.extend(parse_adoc_file(os.path.join('git/Documentation', f)))

print(len(all_chunks), "total chunks across all commands")

# sanity check: distribution of section types
from collections import Counter
section_counts = Counter(c['metadata']['section'] for c in all_chunks)
print(section_counts.most_common(15))

1135 total chunks across all commands
[('NAME', 167), ('SYNOPSIS', 167), ('DESCRIPTION', 167), ('OPTIONS', 145), ('EXAMPLES', 72), ('SEE ALSO', 72), ('CONFIGURATION', 50), ('OUTPUT', 17), ('DISCUSSION', 16), ('COMMANDS', 15), ('BUGS', 10), ('NOTES', 10), ('CAVEATS', 9), ('FILES', 8), ('ENVIRONMENT', 6)]


In [8]:
!pip install -q sentence-transformers chromadb

In [9]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [10]:
import chromadb

client = chromadb.Client()
collection = client.get_or_create_collection(name="git_docs")

texts = [c["text"] for c in all_chunks]
metadatas = [c["metadata"] for c in all_chunks]
ids = [f"{c['metadata']['command']}_{c['metadata']['section']}_{i}" for i, c in enumerate(all_chunks)]

embeddings = model.encode(texts, show_progress_bar=True, batch_size=64)

collection.add(
    embeddings=embeddings.tolist(),
    documents=texts,
    metadatas=metadatas,
    ids=ids
)

print(collection.count(), "chunks added to Chroma")

Batches:   0%|          | 0/18 [00:00<?, ?it/s]

1135 chunks added to Chroma


In [11]:
import chromadb
import os

# Define a persistent directory for ChromaDB
CHROMA_PERSIST_PATH = './chroma_db_git_docs'

# Initialize a persistent client
# If the directory exists, it will load the existing database
# If not, it will create a new one
client = chromadb.PersistentClient(path=CHROMA_PERSIST_PATH)

# Try to get the collection, create it if it doesn't exist
try:
    collection = client.get_collection(name="git_docs")
    print(f"Loaded existing collection '{collection.name}' with {collection.count()} items.")
except:
    collection = client.create_collection(name="git_docs")
    print(f"Created new collection '{collection.name}'.")

# Now, re-add your chunks to the (potentially new or empty) collection.
# You might want to add logic here to only add new chunks or update existing ones.
# For simplicity, we'll clear and re-add for demonstration, but in a real pipeline
# you'd implement incremental updates.

# If the collection is empty, populate it
if collection.count() == 0:
    print("Collection is empty, populating with all_chunks...")
    texts = [c["text"] for c in all_chunks]
    metadatas = [c["metadata"] for c in all_chunks]
    ids = [f"{c['metadata']['command']}_{c['metadata']['section']}_{i}" for i, c in enumerate(all_chunks)]

    # Make sure 'model' (SentenceTransformer) is loaded before encoding
    # This assumes 'model' is defined in an earlier cell and available in the kernel
    if 'model' not in locals():
        from sentence_transformers import SentenceTransformer
        model = SentenceTransformer('all-MiniLM-L6-v2')

    embeddings = model.encode(texts, show_progress_bar=True, batch_size=64)

    collection.add(
        embeddings=embeddings.tolist(),
        documents=texts,
        metadatas=metadatas,
        ids=ids
    )
    print(f"Populated collection with {collection.count()} chunks.")
else:
    print(f"Collection already contains {collection.count()} chunks. Skipping re-population.")

# To make sure changes are saved, you usually don't need to do anything explicit with PersistentClient
# as it saves continuously or upon client shutdown.
# However, if you need to ensure it's fully written for transfer, you can try client.persist() if available
# client.persist() # Not always needed, depends on ChromaDB version

Loaded existing collection 'git_docs' with 1668 items.
Collection already contains 1668 chunks. Skipping re-population.


In [14]:
query = "how do I undo my last commit"
query_embedding = model.encode([query])

results = collection.query(
    query_embeddings=query_embedding.tolist(),
    n_results=5
)

for doc, meta, dist in zip(results['documents'][0], results['metadatas'][0], results['distances'][0]):
    if meta.get('source') == 'progit':
        print(f"[{meta.get('source')} / {meta.get('chapter')} / {meta.get('heading')}] (dist={dist:.3f})")
    else:
        print(f"[{meta.get('command')} / {meta.get('section')}] (dist={dist:.3f})")
    print(doc[:200])
    print()

[progit / 07-git-tools / Undoing Merges] (dist=0.638)
Undoing Merges
Now that you know how to create a merge commit, you'll probably make some by mistake.
One of the great things about working with Git is that it's okay to make mistakes, because it's pos

[progit / 02-git-basics / Undoing Things] (dist=0.693)
Undoing Things
At any stage, you may want to undo something.
Here, we'll review a few basic tools for undoing changes that you've made.
Be careful, because you can't always undo some of these undos.
T

[progit / 02-git-basics / If you would like to keep the changes you've made to that file but still need to get it out of the way for now, we'll go over stashing and branching in <<ch03-git-branching#ch03-git-branching>>; these are generally better ways to go.] (dist=0.729)
If you would like to keep the changes you've made to that file but still need to get it out of the way for now, we'll go over stashing and branching in <<ch03-git-branching#ch03-git-branching>>; these

[git-revert

In [15]:
!git clone --depth 1 https://github.com/progit/progit2.git

fatal: destination path 'progit2' already exists and is not an empty directory.


In [16]:
import os
book_dir = 'progit2/book'
for root, dirs, files in os.walk(book_dir):
    print(root, len(files), "files")

progit2/book 9 files
progit2/book/09-git-and-other-scms 0 files
progit2/book/09-git-and-other-scms/sections 7 files
progit2/book/07-git-tools 1 files
progit2/book/07-git-tools/callouts 20 files
progit2/book/07-git-tools/sections 15 files
progit2/book/03-git-branching 0 files
progit2/book/03-git-branching/sections 6 files
progit2/book/01-introduction 0 files
progit2/book/01-introduction/sections 7 files
progit2/book/05-distributed-git 0 files
progit2/book/05-distributed-git/sections 3 files
progit2/book/04-git-server 0 files
progit2/book/04-git-server/sections 9 files
progit2/book/A-git-in-other-environments 0 files
progit2/book/A-git-in-other-environments/sections 8 files
progit2/book/08-customizing-git 0 files
progit2/book/08-customizing-git/sections 4 files
progit2/book/10-git-internals 0 files
progit2/book/10-git-internals/sections 8 files
progit2/book/B-embedding-git 0 files
progit2/book/B-embedding-git/callouts 20 files
progit2/book/B-embedding-git/sections 5 files
progit2/book/06

In [17]:
import re

def parse_progit_file(filepath, book_root='progit2/book'):
    with open(filepath, 'r', errors='ignore') as f:
        text = f.read()

    # Clean noise
    text = re.sub(r'\(\(.*?\)\)', '', text)              # index terms
    text = re.sub(r'\[\[.*?\]\]', '', text)                   # anchors
    text = re.sub(r'image::.*?\[.*?\]', '', text)             # image refs
    text = re.sub(r'^\.[A-Z].*$', '', text, flags=re.MULTILINE)  # caption lines


    rel_path = os.path.relpath(filepath, book_root)
    parts = rel_path.split(os.sep)
    chapter = parts[0] if len(parts) > 0 else "unknown"
    section_file = os.path.splitext(parts[-1])[0]


    pattern = re.compile(r'^(={2,5})\s+(.+)$', re.MULTILINE)
    matches = list(pattern.finditer(text))

    chunks = []
    if not matches:

        body = text.strip()
        if body:
            chunks.append({
                "text": body,
                "metadata": {"chapter": chapter, "section_file": section_file, "heading": section_file, "source": "progit"}
            })
        return chunks

    for i, m in enumerate(matches):
        heading = m.group(2).strip()
        start = m.end()
        end = matches[i+1].start() if i + 1 < len(matches) else len(text)
        body = text[start:end].strip()
        if body:
            chunks.append({
                "text": f"{heading}\n{body}",
                "metadata": {"chapter": chapter, "section_file": section_file, "heading": heading, "source": "progit"}
            })

    return chunks

In [18]:
all_progit_chunks = []

for root, dirs, files in os.walk(book_dir):
    for file in files:
        if file.endswith('.asc'):
            filepath = os.path.join(root, file)
            chunks = parse_progit_file(filepath, book_root=book_dir)
            all_progit_chunks.extend(chunks)

print(f"Total initial chunks: {len(all_progit_chunks)}")

Total initial chunks: 533


In [19]:
!pip install -q sentence-transformers

In [20]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [21]:
# The 'client' object (chromadb.PersistentClient) is already initialized in cell 6fc11324.
# The 'model' object (SentenceTransformer) is also initialized in cell FzsBvoqh5Kgo.

# Get the existing persistent collection. It should now contain chunks from both 'git' and 'progit2' if this cell is run.
collection = client.get_or_create_collection(name="git_docs")

texts = [c["text"] for c in all_progit_chunks]
metadatas = [c["metadata"] for c in all_progit_chunks]
# Ensure IDs are unique across both datasets. Using 'chapter_section_file_index' for progit2 should prevent collisions.
ids = [f"progit_{c['metadata']['chapter']}_{c['metadata']['section_file']}_{i}" for i, c in enumerate(all_progit_chunks)]

print(f"Encoding {len(texts)} chunks from progit2 documentation...")
embeddings = model.encode(texts, show_progress_bar=True, batch_size=64)

# Add the new chunks to the persistent collection
collection.add(
    embeddings=embeddings.tolist(),
    documents=texts,
    metadatas=metadatas,
    ids=ids
)

print(f"Added {len(texts)} chunks from progit2 to Chroma. Total chunks in collection: {collection.count()}")

Encoding 533 chunks from progit2 documentation...


Batches:   0%|          | 0/9 [00:00<?, ?it/s]

Added 533 chunks from progit2 to Chroma. Total chunks in collection: 1668


In [22]:
import requests
import time

def fetch_git_questions(pages=5, pagesize=100):
    all_questions = []
    for page in range(1, pages + 1):
        response = requests.get(
            "https://api.stackexchange.com/2.3/questions",
            params={
                "page": page,
                "pagesize": pagesize,
                "order": "desc",
                "sort": "votes",
                "tagged": "git",
                "site": "stackoverflow",
                "filter": "withbody"
            }
        )
        data = response.json()
        all_questions.extend(data.get("items", []))
        if not data.get("has_more", False):
            break
        time.sleep(1)  # be polite to the API, avoid rate limiting
    return all_questions

questions = fetch_git_questions(pages=5)
print(len(questions), "questions fetched")
print(questions[0].keys())

500 questions fetched
dict_keys(['tags', 'owner', 'is_answered', 'view_count', 'protected_date', 'accepted_answer_id', 'answer_count', 'community_owned_date', 'score', 'last_activity_date', 'creation_date', 'last_edit_date', 'question_id', 'content_license', 'link', 'title', 'body'])


In [23]:
def fetch_answers(question_ids, batch_size=30):
    all_answers = {}
    for i in range(0, len(question_ids), batch_size):
        batch = question_ids[i:i+batch_size]
        ids_str = ";".join(str(qid) for qid in batch)
        response = requests.get(
            f"https://api.stackexchange.com/2.3/questions/{ids_str}/answers",
            params={
                "order": "desc",
                "sort": "votes",
                "site": "stackoverflow",
                "filter": "withbody"
            }
        )
        data = response.json()
        for a in data.get("items", []):
            qid = a["question_id"]
            # keep only the top-voted answer per question
            if qid not in all_answers or a["score"] > all_answers[qid]["score"]:
                all_answers[qid] = a
        time.sleep(1)
    return all_answers

question_ids = [q["question_id"] for q in questions]
answers = fetch_answers(question_ids)
print(len(answers), "answers fetched")

466 answers fetched


In [24]:
from bs4 import BeautifulSoup

def clean_html(raw_html):
    soup = BeautifulSoup(raw_html, "html.parser")
    return soup.get_text(separator="\n").strip()

so_chunks = []
for q in questions:
    qid = q["question_id"]
    if qid not in answers:
        continue  # skip questions with no fetched answer

    a = answers[qid]
    question_text = clean_html(q["body"])
    answer_text = clean_html(a["body"])

    combined_text = f"Question: {q['title']}\n{question_text}\n\nAnswer:\n{answer_text}"

    so_chunks.append({
        "text": combined_text,
        "metadata": {
            "source": "stackoverflow",
            "question_id": qid,
            "title": q["title"],
            "question_score": q["score"],
            "answer_score": a["score"],
            "tags": ",".join(q.get("tags", [])),
            "link": q["link"]
        }
    })

print(len(so_chunks), "Q&A chunks built")

466 Q&A chunks built


In [38]:
!pip install -q langchain-text-splitters

In [40]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)

final_so_chunks = []
for c in so_chunks:
    sub_texts = splitter.split_text(c["text"])
    for sub in sub_texts:
        final_so_chunks.append({"text": sub, "metadata": c["metadata"]})

print(len(final_so_chunks), "final SO chunks after splitting")

1326 final SO chunks after splitting


In [ ]:
texts = [c["text"] for c in final_so_chunks]
metadatas = [c["metadata"] for c in final_so_chunks]
ids = [f"so_{c['metadata']['question_id']}_{i}" for i, c in enumerate(final_so_chunks)]

embeddings = model.encode(texts, show_progress_bar=True, batch_size=64)

collection.add(
    embeddings=embeddings.tolist(),
    documents=texts,
    metadatas=metadatas,
    ids=ids
)

print(collection.count(), "total chunks in collection now")

Batches:   0%|          | 0/21 [00:00<?, ?it/s]